# 🧪 시나리오 1: Azure OpenAI 백엔드 연결 테스트

## 목적
APIM을 통해 Azure OpenAI 백엔드가 정상적으로 연결되었는지 검증합니다:

1. **기본 연결** — APIM → Azure OpenAI Chat Completion 호출 성공 여부
2. **인증 검증** — Managed Identity 인증이 올바르게 동작하는지
3. **응답 구조** — Azure OpenAI 응답 포맷(usage, model 등) 확인
4. **에러 처리** — 잘못된 키, 없는 모델 등 에러 시나리오

## 사전 조건
- Lab 1 완료 (APIM 배포)
- Azure OpenAI 리소스 생성 + gpt-4o-mini 모델 배포
- APIM에 Azure OpenAI API 등록
- Managed Identity 역할 할당 완료

In [4]:
import os
import requests
import time
import json
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)

APIM_URL = os.getenv("APIM_URL")
SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다. .env를 확인하세요."
assert SUBSCRIPTION_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다. .env를 확인하세요."

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
HEADERS = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
}

print("✅ 환경 설정 완료")
print(f"   APIM URL:    {APIM_URL}")
print(f"   Deployment:  {DEPLOYMENT_NAME}")
print(f"   API Version: {API_VERSION}")

✅ 환경 설정 완료
   APIM URL:    https://apim-ai-gw-aigateway-20260317.azure-api.net
   Deployment:  gpt-4.1-nano
   API Version: 2025-04-01-preview


---
## Test 1: 기본 Chat Completion 호출

APIM을 통해 Azure OpenAI에 간단한 요청을 보내고 정상 응답(200)을 확인합니다.

In [6]:
print("▶ Test 1: 기본 Chat Completion 호출\n")

start = time.time()
resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers=HEADERS,
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "안녕하세요! 간단히 인사해주세요."}
        ],
        "max_tokens": 50
    },
    timeout=30
)
elapsed = int((time.time() - start) * 1000)

if resp.status_code == 200:
    body = resp.json()
    answer = body["choices"][0]["message"]["content"]
    print(f"  ✅ 성공 (HTTP {resp.status_code}, {elapsed}ms)") 
    print(f"  📝 응답: {answer}")
else:
    print(f"  ❌ 실패 (HTTP {resp.status_code}, {elapsed}ms)")
    print(f"  📝 에러: {resp.text[:300]}")

▶ Test 1: 기본 Chat Completion 호출

  ✅ 성공 (HTTP 200, 5563ms)
  📝 응답: 안녕하세요! 반갑습니다. 어떻게 도와드릴까요?


---
## Test 2: 응답 구조 검증

Azure OpenAI 응답에 포함되어야 하는 필드들을 확인합니다:
- `id`, `model`, `choices`, `usage` (prompt_tokens, completion_tokens, total_tokens)

In [13]:
print("▶ Test 2: 응답 구조 검증\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers=HEADERS,
    json={
        "messages": [{"role": "user", "content": "Say OK"}],
        "max_tokens": 5
    },
    timeout=30
)

if resp.status_code != 200:
    print(f"  ❌ 호출 실패: HTTP {resp.status_code}")
else:
    body = resp.json()
    
    # 필수 필드 체크
    checks = {
        "id": "id" in body,
        "model": "model" in body,
        "choices": "choices" in body and len(body["choices"]) > 0,
        "usage": "usage" in body,
        "usage.prompt_tokens": body.get("usage", {}).get("prompt_tokens") is not None,
        "usage.completion_tokens": body.get("usage", {}).get("completion_tokens") is not None,
        "usage.total_tokens": body.get("usage", {}).get("total_tokens") is not None,
        "choices[0].message": body.get("choices", [{}])[0].get("message") is not None,
        "choices[0].finish_reason": body.get("choices", [{}])[0].get("finish_reason") is not None,
    }
    
    all_pass = True
    for field, ok in checks.items():
        icon = "✅" if ok else "❌"
        print(f"  {icon} {field}")
        if not ok:
            all_pass = False
    
    print()
    usage = body.get("usage", {})
    print(f"  📊 토큰 사용량:")
    print(f"     Prompt:     {usage.get('prompt_tokens', 'N/A')}")
    print(f"     Completion: {usage.get('completion_tokens', 'N/A')}")
    print(f"     Total:      {usage.get('total_tokens', 'N/A')}")
    print(f"     Model:      {body.get('model', 'N/A')}")
    
    print(f"\n  {'✅ 모든 필드 정상' if all_pass else '❌ 일부 필드 누락'}")

▶ Test 2: 응답 구조 검증

  ✅ id
  ✅ model
  ✅ choices
  ✅ usage
  ✅ usage.prompt_tokens
  ✅ usage.completion_tokens
  ✅ usage.total_tokens
  ✅ choices[0].message
  ✅ choices[0].finish_reason

  📊 토큰 사용량:
     Prompt:     9
     Completion: 2
     Total:      11
     Model:      gpt-4.1-nano-2025-04-14

  ✅ 모든 필드 정상


---
## Test 3: 응답 헤더 확인

APIM이 추가하는 헤더와 Azure OpenAI가 반환하는 rate limit 관련 헤더를 확인합니다.

In [14]:
print("▶ Test 3: 응답 헤더 확인\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers=HEADERS,
    json={
        "messages": [{"role": "user", "content": "Hi"}],
        "max_tokens": 5
    },
    timeout=30
)

# 주요 헤더 확인
important_headers = [
    "x-ms-region",                     # Azure OpenAI 리전
    "x-ratelimit-remaining-tokens",     # 남은 토큰 할당량
    "x-ratelimit-remaining-requests",   # 남은 요청 할당량
    "x-backend-url",                    # APIM이 라우팅한 백엔드 (정책 설정 시)
    "x-request-id",                     # 요청 추적 ID
    "apim-request-id",                  # APIM 고유 요청 ID
]

print(f"  HTTP Status: {resp.status_code}\n")
print(f"  {'헤더':<40} {'값'}")
print(f"  {'─' * 60}")

for h in important_headers:
    val = resp.headers.get(h, "(없음)")
    icon = "✅" if val != "(없음)" else "⬜"
    print(f"  {icon} {h:<38} {val}")

# 기타 x- 헤더
print(f"\n  기타 x- 헤더:")
for k, v in resp.headers.items():
    if k.lower().startswith("x-") and k.lower() not in [h.lower() for h in important_headers]:
        print(f"  ⬜ {k:<38} {v}")

▶ Test 3: 응답 헤더 확인

  HTTP Status: 200

  헤더                                       값
  ────────────────────────────────────────────────────────────
  ✅ x-ms-region                            East US
  ✅ x-ratelimit-remaining-tokens           4967
  ✅ x-ratelimit-remaining-requests         4
  ⬜ x-backend-url                          (없음)
  ✅ x-request-id                           0c602cb5-a17a-4494-90de-26c4c702c48e
  ✅ apim-request-id                        4bb872be-0864-445d-91a4-a1b54b769953

  기타 x- 헤더:
  ⬜ x-accel-buffering                      no
  ⬜ x-ms-rai-invoked                       true
  ⬜ x-ratelimit-limit-requests             5
  ⬜ x-ratelimit-limit-tokens               5000
  ⬜ x-ratelimit-reset-requests             12000
  ⬜ x-ratelimit-reset-tokens               396
  ⬜ x-ms-client-request-id                 Not-Set
  ⬜ x-ms-deployment-name                   gpt-4.1-nano
  ⬜ X-Content-Type-Options                 nosniff


---
## Test 4: 인증 에러 시나리오

잘못된 Subscription Key, 키 없이 호출 등 인증 실패 시나리오를 확인합니다.

In [15]:
print("▶ Test 4: 인증 에러 시나리오\n")

payload = {
    "messages": [{"role": "user", "content": "test"}],
    "max_tokens": 5
}

# 4-1. 잘못된 Subscription Key
print("  4-1. 잘못된 Subscription Key (→ 401 기대)")
bad_headers = {"Content-Type": "application/json", "Ocp-Apim-Subscription-Key": "invalid-key-12345"}
resp = requests.post(BASE_URL, params={"api-version": API_VERSION}, headers=bad_headers, json=payload, timeout=10)
icon = "✅" if resp.status_code == 401 else "❌"
print(f"  {icon} HTTP {resp.status_code} (기대: 401)\n")

# 4-2. Subscription Key 없이
print("  4-2. Subscription Key 없이 (→ 401 기대)")
no_key_headers = {"Content-Type": "application/json"}
resp = requests.post(BASE_URL, params={"api-version": API_VERSION}, headers=no_key_headers, json=payload, timeout=10)
icon = "✅" if resp.status_code == 401 else "❌"
print(f"  {icon} HTTP {resp.status_code} (기대: 401)\n")

# 4-3. 존재하지 않는 배포 이름
print("  4-3. 존재하지 않는 배포 이름 (→ 404 기대)")
bad_url = f"{APIM_URL}/openai/deployments/nonexistent-model/chat/completions"
resp = requests.post(bad_url, params={"api-version": API_VERSION}, headers=HEADERS, json=payload, timeout=10)
icon = "✅" if resp.status_code == 404 else "⚠️"
print(f"  {icon} HTTP {resp.status_code} (기대: 404)")

▶ Test 4: 인증 에러 시나리오

  4-1. 잘못된 Subscription Key (→ 401 기대)
  ✅ HTTP 401 (기대: 401)

  4-2. Subscription Key 없이 (→ 401 기대)
  ✅ HTTP 401 (기대: 401)

  4-3. 존재하지 않는 배포 이름 (→ 404 기대)
  ✅ HTTP 404 (기대: 404)


---
## Test 5: 다양한 요청 파라미터

`temperature`, `max_tokens`, `top_p` 등 주요 파라미터가 정상 동작하는지 확인합니다.

In [16]:
print("▶ Test 5: 다양한 요청 파라미터\n")

test_cases = [
    {
        "name": "temperature=0 (결정적 응답)",
        "params": {"temperature": 0, "max_tokens": 20},
    },
    {
        "name": "temperature=1.0 (창의적 응답)",
        "params": {"temperature": 1.0, "max_tokens": 20},
    },
    {
        "name": "max_tokens=5 (짧은 응답)",
        "params": {"max_tokens": 5},
    },
    {
        "name": "system + user 메시지",
        "params": {"max_tokens": 30},
        "messages": [
            {"role": "system", "content": "You must reply in Korean only."},
            {"role": "user", "content": "What is APIM?"}
        ],
    },
]

for tc in test_cases:
    messages = tc.get("messages", [{"role": "user", "content": "Say hello"}])
    body = {"messages": messages, **tc["params"]}
    
    start = time.time()
    resp = requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers=HEADERS,
        json=body,
        timeout=30
    )
    ms = int((time.time() - start) * 1000)
    
    if resp.status_code == 200:
        data = resp.json()
        answer = data["choices"][0]["message"]["content"][:60]
        tokens = data.get("usage", {}).get("total_tokens", "?")
        print(f"  ✅ {tc['name']}")
        print(f"     → {answer}  ({tokens} tokens, {ms}ms)\n")
    else:
        print(f"  ❌ {tc['name']} — HTTP {resp.status_code} ({ms}ms)\n")
    
    time.sleep(0.5)

▶ Test 5: 다양한 요청 파라미터

  ✅ temperature=0 (결정적 응답)
     → Hello!  (12 tokens, 1711ms)

  ✅ temperature=1.0 (창의적 응답)
     → Hello! How can I assist you today?  (19 tokens, 1863ms)

  ✅ max_tokens=5 (짧은 응답)
     → Hello!  (12 tokens, 1895ms)

  ✅ system + user 메시지
     → APIM은 "API 관리"를 의미하는 "API Management"의 약자로, 애플리케이션 프로그래밍 인터페  (53 tokens, 1821ms)



---
## Test 6: 연속 호출 안정성

5회 연속 호출하여 일관된 응답과 안정적인 지연 시간을 확인합니다.

In [17]:
print("▶ Test 6: 연속 호출 안정성 (5회)\n")

TOTAL = 5
latencies = []

print(f"  {'#':>3}  {'Status':>6}  {'Tokens':>6}  {'Latency':>8}  Response")
print(f"  {'─' * 60}")

for i in range(1, TOTAL + 1):
    start = time.time()
    resp = requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers=HEADERS,
        json={
            "messages": [{"role": "user", "content": f"Reply with the number {i}"}],
            "max_tokens": 5
        },
        timeout=30
    )
    ms = int((time.time() - start) * 1000)
    
    if resp.status_code == 200:
        data = resp.json()
        answer = data["choices"][0]["message"]["content"].strip()[:30]
        tokens = data.get("usage", {}).get("total_tokens", 0)
        latencies.append(ms)
        print(f"  {i:3d}  {'✅':>6}  {tokens:>6}  {ms:>6}ms  {answer}")
    else:
        print(f"  {i:3d}  {'❌':>6}  {'—':>6}  {ms:>6}ms  HTTP {resp.status_code}")
    
    time.sleep(0.5)

if latencies:
    import statistics
    print(f"\n  ─────────────────────────────")
    print(f"  성공률: {len(latencies)}/{TOTAL}")
    print(f"  평균 지연: {int(statistics.mean(latencies))}ms")
    print(f"  최소/최대: {min(latencies)}ms / {max(latencies)}ms")

▶ Test 6: 연속 호출 안정성 (5회)

    #  Status  Tokens   Latency  Response
  ────────────────────────────────────────────────────────────
    1       ✅      15    1818ms  1
    2       ✅      15    1784ms  2
    3       ✅      15    1659ms  3
    4       ✅      15    1867ms  4
    5       ✅      15    1786ms  5

  ─────────────────────────────
  성공률: 5/5
  평균 지연: 1782ms
  최소/최대: 1659ms / 1867ms


---
## ✅ 체크리스트

| # | 확인 항목 | 기대 결과 |
|---|-----------|----------|
| 1 | Chat Completion 기본 호출 | HTTP 200 + 정상 응답 |
| 2 | 응답에 usage (prompt/completion/total tokens) 포함 | 모든 필드 존재 |
| 3 | 응답 헤더에 x-ms-region, apim-request-id 존재 | 헤더 확인 |
| 4 | 잘못된 키 → 401, 없는 모델 → 404 | 에러 코드 일치 |
| 5 | temperature, max_tokens 등 파라미터 정상 동작 | 200 + 파라미터 반영 |
| 6 | 연속 5회 호출 안정성 | 5/5 성공, 일관된 지연 |